# MandiaCompact — étape 7 : reformulations françaises ancrées dans les sources

Calcul privé du projet https://github.com/mandi-model/mandia-compact ; fiche officielle de type modèle https://dataset.mandia.org/gnakalehacker/mandia-compact.

Deux bras de 400 mises à jour depuis le checkpoint CE sélectionné étape 6 : contexte/question canonique ou enrichissement français par l'assistant. 40 familles de réponses baoulé copiées de Wiktionary/Kaikki (CC-BY-SA-4.0), toutes issues du TRAIN de l'étape 5 ; 120 vues train, 40 validation, 40 test. Les références sont déjà connues : seuls la formulation française et le format question avec contexte changent. Ce n'est pas un test de nouvelles connaissances ou de conversation libre.

Les deux bras tirent les mêmes identifiants et les mêmes cibles. Leurs nombres de tokens source peuvent différer ; temps et tokens réels sont comptés séparément. Sélection gelée sur validation avec garde de régression avant tout test. 13 782 528 paramètres inchangés. AdamW/RNG/compteurs sauvegardés dans Drive tous les 100 pas. Les parents étapes 4–6 doivent être présents dans le même Drive. Aucune donnée baoulé inventée, aucun logits/poids privés de l'assistant, aucune validation native.

Méthode : augmentation de données française assistée par grand modèle, et non distillation de poids. Références : https://aclanthology.org/D16-1139/ et https://arxiv.org/abs/2305.02301. Les résultats de ces articles ne sont pas des résultats Mandia. Ce lot n'est pas un nouveau corpus de conversations réelles.


In [ ]:
# Mandia stage7: PRIVATE compute, public project is GitHub / model card.
from google.colab import drive
import pathlib,sys,base64,gzip,hashlib,json,subprocess
try:
 import tokenizers,sacrebleu
except ImportError:
 subprocess.check_call([sys.executable,'-m','pip','install','-q','tokenizers==0.22.1','sacrebleu==2.5.1'])
if not pathlib.Path('/content/drive/MyDrive').exists():drive.mount('/content/drive')
ROOT7=pathlib.Path('/content/drive/MyDrive/Mandia/Sauvegarde-langues-MandiaCompact-2026-10-13/colab-runtime/mandia-compact-s2s-etape7')
ROOT7.mkdir(parents=True,exist_ok=True)
ASSETS7={'mandia_stage7.py': 'H4sIAAAAAAAC/51aa2/juJL97l/B2xcLSR1GsT22k7ZHwGZ7PIPgdqd7k+zcBbyGQEu0rY1eLcpJPEH++54iKdnOY3JngenYJotksZ6nivPhw4cbKaK1rI6FUomqZcx+rWQerZnYrDKZ16JOipyzZfKAqeI+Z72f/NOzrywrYpn6nX/kGDuuRbWSNSvXlVBJvmKqTJOas8tvN2yTKylbiqJiOXa8kywq8jtZKb29SJm8E+lG//A7Hz586CRZWVTYUdTrNFnw/1VgYi2U/lGJPC4yvop4VJTbhrQuqmjd/MhAkohQ1WIlB0wopgavTQ311PC1qZGeGnWWVZG1E/JHH/+YJf+qR6/71xz/Phf5Mll1rr59uwnUwKdPvxQQZX3imOXHUZGVIqqPVV8dy1qU8tTpXN9Mv1/zL1fBoNvlfXk87HRiuWTQAOTrVsW94nVxyyORF3kSiTT4VaRKeuMOq6TapHXwqMazOVtCsIolOZs5dSWS3OEOBJrEWqL4UUtVO/OnjiasiJC2xi6sxP3KOqhmTntGaMacOUuWrB1luAQDmdatMw8CexKTYIgmmlXYNKllFjw6qthUkXTGs8H8CLfw7a0MoecnsTqa9efgThuHM96jwX4iV/eycuZ7hGDaGcdJVLsVt4zbzXA1KExJqCWVuUsMzJrz597PQW/U1RfYTdpD9eSnPseBSayZN4Kd7a4690VZyjzWCz2SfL2pckvX6fy7tjw/yZeSPEeG5Bqup/X4Q4RpoZSrvYXTesVjeZdEWoXGh8j0XW9SFzXUCxHIXAVdoylYIi5E2hL5Srpd3rCvPN4b0RZM8AXf8pBsbiHqaG2mZ3rlWP896o3mzaFYcJ/Ua7iDDwes5UPteuM7ntPyPUb1pkSsmToKlmkhaveOmCT2joK8lcKjQ+tId6A8MfNaURjb49aKOzQEWtX4fDLWjkCQLLdhKZJKxqESGeSer1zisCpSXpsIRdcVgRH2bzKXlcB31/PhXRsYLcJM7A763mTxPo2RbrKTrHZDLc8HEkW0LgolWwZ6Iy68yXZvxrJEMwtvZ3pkNGSjc2NMO297gL+8ObmdH+7QWObL5S9naG2rCUhOhnEl7vMwl/chDgvhOiHsPjQrIfibaiO5k+Tlpg6hn1W9VmEmtmGcLGG/zXyjg7DI022I4I8TpJm0Osvg+65WiWHcyDzaxMJPcOSdSFKxSOEH2u0OZxfL3ihUm5KiKBQCARrztIozP1yHyB1vAqn/Mv394vM0MBMTQ6VgTfkmC+t1JUWs3IE30UE3u42TypUPSGZhcRsQyziAgmiRxqTCspIUmOnYskqKakjeM2wCz4xSDXyBtkw9LeiUBO2ChPY/cX5sijqJEyQ1u5NPS1LH84mR0DiVr2MHRCiV680phjfHjei40fvHjexxyBxpsdpI9ZePIycM3jzA7G4N+XhVFZs8lvHxD/H+/hCnWRY8y1PexLrM8wlu1WCNRaSpK3bmHASLZ7aNAERM/pGUjRM2iW3ehIN2wCPJ6kwbtEJDFLbrTkyQRWJqT4cNqLVwzRoPiUz0xVIOe/2zeLQQw+VZ91R246h/thDLZbwcdUen3bNFLD+NRr3uYPTp7KfT+FPcO4ukOIsWfRE7xMCWZGwtmL7a/XkmSuSASGfiwInKjcPvZbJa10o7V3AoGbsPLkcBMvlDViF47Q9HJCXLuIUXJzsarTKHdknTEElfBLOWCGxgVKbPzIe3BBhOkxfmtZun/U7uxYNIjxei2KTymGyiJXvPLfi7lszfM0UYnHwokSQIkBKuIIDmjE+5Y2SsM27aCGp8qF/+UpTjP5Ey1/dtKR9VXbklXCHVsBX5y92Ddp7XHmZcqySrbZTwxB1ygt1epZ8jQr+2JEW0cmEprgW8/nd88lmI0JvKMISNH2mS/WkKEQ2BZxnyV2mxcJ2Pfrl1PA8MANGXKoTsQlFlztjATSdFoP9yhVnkQ2c86HPnHtObEt+73BEwYaTtyMR8Z4NqgJJkA//56VmfD/tn5HS4Tg0U71BGSWroN/nDQM4x9k6xBVZZOP15yoiZHnAuWwI7rdl5LLJ/MvDGiDcSFjxWKC1n4yLOuMsbu0A6S4kdW55Y+IKszn7ApOhQdUIc6SJEKkbONUHBkawSKjD+Q5su7AgInBlgSD+Q5qSasLxoQhqDiSa1OrE+elLZCkUqzeEmpyQK0E3Y5s5oUPE70qHdwNeIxyXhW8yzFFlCHmakqyNpqFC60G3WMo2Pi03dVF13ibwH70t2ff51yv5x+e2fl8zWTs02nAyM2frl5ur84pL4r59VWruCCnxnyQPO8of7UvvPc4AX5vcZ0iK7/vrtC6NAsD3DWG+oB1FkIcDqgeY46CdXqd6WxruGUC8cONzq3OjfH+EItqtCyACO6DxblC1EKoCW48Px0eGKCYEbMNEb4BwbqCZ05ufpz4E1uSOwMbGrD4Z73QnQbg4UCIMr/pA5W0joSzJdDUEqhJcBOBebWJcfGj8xDX3AVhLDn7ATTIkQDOCKrO4SJQnMEHadMFgSCdIakxllBkfpJfBIlkmh4EYx1yoSzJ7JqBbc1JLt+xrYgwlkoroFc6ZMDvequLEu/bizQgoZ7wEqMjGDjEIKL25XhzxMIxIieBnKMKRqG9sgVqBWQggCMs6SKKQIa7HA9L+/T68uvk4vb0w+4bug672x5Hcs+PXi8/nNxbfLY5js9y8Xl7/Z1X+O6v88n/+dXckNysp6LYGacLlio9KtKRQSiE+78TFVYNCYXgeZU2VDriRjqC1foQpVjeKNjvwOmU6YVgQUvlxN9N/gyxXwTLUdv3/Bq+n3q4vrqb0fVpsrusTxkqJMil30nuYYwrXLxqClm/GFMLU7+xG0leHu/nu6bgu2SUFZ3zozbYGdX6ec7byPogtC93xS0kXpGEkHGfj51mpdxs0nMWHhWGaFOUuDt4ami9JFSz7UZmyrZVt84EL7xvqDOySE/aGCm8w9OhgtDaGOIaHeHSU+d2SKwL2giB8HQU/7k0viY4lil0UuqZXkFj8HNDZ7ftScYoJeUzYUL0+eU4TwPP5CbmN/9PFHI5Ajv/+xoD/lk1EnAYfQZFtd/9ja5nkVuuege1MhIV+aNj0sf1eQsixo+0lu209yP35ssUqkR8g/fKCQxj4yDTapZ1Wj9qPWSLtgN4ZFE6uorEPZVgWPthlEIYMT5CpN/8jd6wNZV3W4/eJxt83GS52vnLY4n5MoSD6qlJFyEy0am0ICmA1cZRuLLSWJYlkDbkh8XRWgxv5OinoNJ6I2Tv5tQGbV2NXMNbztwgWylj/kpg3G3cam7exMo9ktyM6432+ptNM0JIYnTdF7ttFwb6Mm1UmzWe/1zdrzBhwmZ2nmWp0HhjJRyQohHVE+sG1M38BCV9dn8SYrlfvo7CKuM959J0gG8EaqeuIKRXN4K7fKVA5NIQmrWMuHOFkht1Fp+1ak06HORMyYJvVX+LpWG29ilYHk2jZaxm0Y4F2fa1QZGDCpNRsMu3Tm60FwL3Vor3DbzU+ctjrjj42Bj19YPN835TG8ZmfsfxXeP+luDRxgRufPUUxoASBdOI1UniavZQLDriZB3qHOhU0Du8087QEpyyaryI+KlKKKexAJZFbW2zAij3F3NR9Byce72Y4TmNUObjQtM1MYa6RJR/qUEqgaf0IF2+uwaF0omQcZlEnTHBYSpCJbxEK7+HjH5uFBC9hLky/eyvLX0y/TzzoDnv9+fnlzfDO9biDCo2MQljM2DEBVtBdh5Nvx3dtH6cvc8t119lAzkQEthQAAse05eSb20pSb6dDStL+Rd5BsXiZT0/FuUpfJMQZ56gLxeUZ9k3wI8qFP7TR7Qhso3lpBFegrWfc5vbW23kAf8GbGfdp1pJ2V6WiaxL37oXRjgOQa7gZfkcVuY/5pRMb6d3ahQW60X0SZBrGBsZikyjtNoqRuagc1ZvV9wVQGT+fsdv/9B6dFt8qnBwAU8A/B48q0Oscrre4VKXuP7acJYUMVzCheEsED35oc1Ov1eX/wCcG21zvlvbMzk16YCPTGs2XbLHh8GHcH8dPxANBl8XJ2287q5bb71LxVIIk5c9Pacp3/yb9KpaA/NmaON+uZrtS/Qkhb65s07wSPuDU8YCbM9fnCfCKOJY28w0beTccVwBUFgWr7sNRrbhIx9cqs3KhciRK46t+CA4rFSwruLIp63UAqAh4HK1YvVwTBc4pKNo8ahmKnx5ngi7mnw2ljnjCL0BqJRv2IWYH+3MFEQ7pvu50XmdIUaUSjAuvwr0bkPXdQx+JO5E1EppXe5C9F4gYJNeFo/EoGj/abe6+msH+p1fcKYoueYzXiPTpIVK0TvysWw9S+bIj6/y8ak+yCt7GJ0ZcztnojrALj10mobQKFi+0B6G+TBb1YvMh1YVPnNc6x12ka9346PesP+2cAjUmWUDdm5gy6bRV+2CthIqUu9pZRa4SUa7S538dgbYqCE0JLTSfmvqhiqiKpinil2UQmgyqIUQOHwXl1D+mwBROlIqG+VvugjvIUpTBbbJl5bIfJUl9Az+i4AGEjjMegQcI5eCJXmtC+nR887VItRGXEK00sqpGsAU5QSqNywn/18/f+/Yd+xz4NPetmLGR9TwIkpU7e6GRQ136/mQFASXfda2lQq4hsUe2X9MUCMfmO2K/kCuGBehQ2kegWnn61ftX9K6EfkOxTujVwY626Hk8lUku567Hqvm1lG6SOR4/bJb1GEaV9pnLCsNxqyw9DR9+TTIZ6q7XpAtGQ6eICd8eKUqXr6HnH7GAm/xY41+f/9fv0t/OrX6bHpocwtX2D+Vsg660FwFiNPyCMhBn1IPE1rmAK7XvdO36mpQFPeXSog2waQy+72lQML7Y1UZYaZNPDDyAbEDSOeNZd9552olXmIU+f4mmMDdSDoOl8Pb/85eI8nN6cf5+ehjfTq68Xl1OHH9Q8zwHkYTx5hnocsayJ2KJLHRMPSd5Elk8cFg3LDIWKksT+jxR8mW7UunmEabMUGVEHBhLqlloYBgFMg8pG2MXYPH12/g8Zb7uSPCMAAA==', 'teacher-grounded-qa.jsonl': 'H4sIAAAAAAAC/+3d3Y7jxpUA4Pt9CsLAYG7S7ZaoGc8kF4txj7OwM5O14wwWARYwSlKpu9wUyeZPjzvBAn6IuQnmxneeNvYF9lpX/Rp5kj2nir9ilbrYLYot6gBJ7Ig/RYnn60MeFqv+8ZmYf/Z757OEs9k5j45OTk6eH5189jvnswVbCu8al70XF0IuwI/j0BOJ3CJiwsdPEhZf4AcRD2F9+GAeLHERfHQWBak/5/Mf5gz29cMlw8VhFCxDuYvTwE/4Twl3fu/8KfCZ7zAn9bkzO199uor48X/7b3kcszNc/l3KPYf5Ysm8bN0wiOPVpzk/So6E5/w77nnG/MAXM+b90F0bzI/f8wj3rFbxuDMNPHEsf5sgjWa80riuxcqKaeThWudJEsa///zzRXQMP3UiYKPo+jiIzj7HX/5z3H1lI48vlww3yz/3xIz7MZdf9vToy78dff/qaHJ88gcnjHjMoyvusCSJxDTFPf8OjoP5ZzyGLzp31C6dcxEnQXTteMK/iOW5j7g/O/8hiMSZOpEsjmEd5idHLE3Og4jPnZn6YVP4sb57JfcWsoiF5xGLYe/5mXeEn7fyR7nTPzh+kDi4F+4nYuZEfBZEuOJcMC84S7k8gCkLUo9XDoD/xGaJo/4/tPhfxe/kwJJlCGdhAb+6A8d4xp85f/3Lq6//7AS+ikefJeKK/xDxK8HfQzBOr39YwvEK/BUXzIs5/rizIJQ/4Tn35kdBmmSH68gvJPyzPzjfv3r7lfPXV3/5j6/+6vzx1duv33z91ff49SSEz6+YJ+YMD+rzhMeJ+popnBfuHyUsOuOJc8Z9HsFqf5erffY///YPHb5RX/jyYCeDZPDADY77MghHDSEThJFY3SRMRNyZ/+vnf9pgYSGc+kTACXVWv2R7I5AEchggXRuQZWtbU/mujFGdsDUpQBW+5RGc1suUr2fVWbBc8lwSySSZw5A5sUqV0MTWTJ4qZlnMX6bi7ivXb3n601GSQoitbmYCogyBFnhwZ5QzSeaQZLpjQxXHHXd4EfsGafoLOL0QqwsmljUWp5gBYVHM4T/wT69c20bdw3ZecpsytMbecz21ZjOtocG+dc6yj4nZoJiNemUGIaPCnbyRt4PwNu7BmwpMjHX4ZyIDXkQQKfzvlbiv8XgN8a8uMrGQk8hrTPJH/vbfn2vjr4NSTCVOVzewRgqBA7d8usQn77+mPA7g23uNXGn7qIL8kb9H6G9ilf+2WXCB9AchHsujQnIyF6onFasbDH6+MRXmxZf56mYW4SONrD5aXLpSWiSWe87yhana8qLDy9JvnjKBjxXCSMRVA69i5JYtsDLVYk8loD87iagVFMtdV/fXms3aXgs3+ecEZ1BwRj3Agb/tEKBTwf08So8JEAHaT0Dj3gDFTMRCXc4JuEpa3SSrGzgt9cu/r9QT8dxACOfWSQJBrIjVo2bl2rDqoM5x2uAk77XyYIf7ryXDU5xoUpYCCScB+5AFlLrI2OM2NrFKXdusZWSZKw/rgoouZWGRAgiwuFi9meqIGBF7vMSenRjqEs9OOrw6fHXNZMpKxFLMubY8h4txNRs9bXdXEsKPEuH41/ARW6S3H/WlvrUGWovC/etE5Z+TqEGJGvUnCrv/hTyKAx/+hXgRrwHyGvfDS3bsgIjJHivLYBVwkdd8sIxtw/0YnOwZLIdLME7WyNpeWnNtrHVQ48AwBTZL+CTSJrLGM2P40TNxn7KX4sgcmdtHcxOr/Lbl/htFOmvkrWZ+k10VeZbm5lp3YibIHtnbM3umThrPunxU5snOUUuh3u1e3SRRIBIThHJlG10P2XVFWiJ88dRHbQuhrzdqG2ptrb73glr2MUkblLTR7qW9qUVpyFPHwyM+JmyEbdjYxrvHdqqL0sa1JPzGkgNTFNPImaeEjtANAZ1rg66L92CKUGUyycl3YCQubabLKihMqGfNlPOI3yD4Taxy3pbLKM0shvG6YGLq3TXgyIyuPEnhsBQ+N/UueX7S6UB5i4VwfgzS/PXPs1TAMdf9vebq5Wvhx0mUSh3ldm1Gq9xiW9UBf3DxNHDOYG+mAX8MLbf2iG3oQOafk8hBiRz1JRJzBs++DsYsxm5ERIkoEV0jOu41acZw/Yr9yHROvltDIjdQ25JFsjhAi66NxS7e8skGsaxFq3zjoDRXY/N9GlG+JKOHaHRilS+3XOrJSeYDd1UozIOaB23hx8eaTKTdkpIqgR0y2JGpKjTq/AL3KhCyBBryKAoARLJhWOZ8/XaJ80EtrOt7n/oOuwg26Wu2134YootAOwyR+pjoDYreqC96DCJ09Sv5I38H7W+8e3+n6r5RRmpZit1kpLou0SN6w6Dn2tDrpkNOEaKS3lWKZ1RTwylfr8BRKJSV6sqEkTAOBOPEKg92MS1QNUizebjyAL6riw5fW588ksdBeDRN6fy8yxkt37JIsCWvzs7lLOE3CHwTjfUtjpIj2YfchuGWWytJ5qvh/Fwp05I0t91+jq5UWyjNPiaUg0I56g8lK2P0mDgSR+JomvW5U47v8shcm102C2HL2Z7zgCeYBHOIMF0bmB1UdvJ4Nc3Krk+gWNspNm096zP5JJ/75nNilTi7KPbkEbs2C7QOZjGklIh4bfrnYjtKo8R0sEy/mBhqQF9MOu8bMEu5wFD3mAM/z1nauK4t1mj7GORee14vtyYiEXLzjRXXZlP3GHo7MQy9ndDLkcPzNurLGz7RCLyEwBG4gwI37gschAucxCJQ5TtW8hl/NNW9YbWI0qL/TRblbV/gIIfk8NE6dG0cdtETpwAoZ0yScSu4sTOOuoc8yiZ1qWxyrx45JJJEPlqRE6vM2El3nMJk2bcmi987uuPInx8T5SVdr5LKwal8MTYUZF6MO57qgsn3/2dsynzeuDaFE8Pl+4O45n2e/G+hhfow4PDpj8I3Dv9dbas1uPqOC2/Zx8RtUNxG/XCrlOnzOD0mdsTuUNiNd8/uXRGaaz1lNBNelD7bzChD1Ija46Pm2lDraD4nUycZc86bgsqmz1aTyxBDYvj4GE6sMl4X1RaMUzVDtQGetsxSQJGbNkASRsK4rxifmYosz3ZQZMHYWIZyDrVA1zNMLX7oHd8DW2mygyh3FoJvtLfWaHuC0IjWYPY5IRwUwlE/CKs9JOsBe0wYCeOBYhzvHuO79Ri9uy7jOcl1KEceVFs+7LqUXJLLx+7StXG54+rNHWmzXjxVLzupLQgn4RwSzolV0uy6prOJo6a0AyRTL0lRc5pteplmc3gXckkqSR2K1NGJYagbWNDlveYFJj9/jr3VAB0OzpQ2O37jRHGJXNeK3H32WQEGH13A5kJ1RE8TQ981QzOtkWFbT07dJ69OddRw6ep/CdvgsI36wYZ5MZjN0nB1g3eDKoAjTvgI3wHhG/eHrzrTm4U+vAKEswLfRe6BzeBWEQ/Q9qUn8kgeH79H18ZjN+89ZaGq3mJSEaxefIKQ1mEsRiEGlFcigTSK9R5Kj8RxOBwnVumxk5LNhSrZ3GGkGJJmdTOLMIHKVxEzh+tZdiGrsqSTdA5A58jw8hMs6PjiFV/Nk6MrhbD5Ogm1sE0WbLO7OjHwdRGy1Dfaqu26vaq1fZeg8gXkaVCeRv14Wk9TMmqjPGyPSRgJG4qw8e6FvVGxKZVhtMJ9mv6mrjqWRXYBqaKbmBGz/WLm2jDrcEruLFDXIGmG4vaEeqEhX+WoVTmTwBG4xwFuYpXXtlknyUoHsewvprtUNEzOZCiNZNecJI/k7ZO8L0w1kC86vmfLJw/D78r1k4O2uUNrs7u6J5zozJ9vqC/Wd90aFOxbxyn7mDANCtOoH0y1ia9JFIkajqjx7kW9ySKzOpGu7hF2PqtCHtJt7rxIF+l6BLpcG13d1jkUteK+qjFH7oYJqymdEbg9AzexSmdb7Q+iRr0tIrzK7q53d0pxhc/yUpPkkby9kffCVOd40fWtGU6PPk8d7ykerJ9o5ulTK7RJZu13Woe19DB8xAUzylpvobUtbEKHK/+cdA1K16gvXaubMwYJK5vqBEP1mIgRsQESG++e2JsClZSGXfRVzFp3ACmBEjyCt5/wXBt4XRZJSoUVVzKGda/VzM6DWD09Zqo3iFyTE0ACuKcAJ1aZb9udQ3wOZyKBTSBcQi+NGw6Blo7h5n4jeeyTRtK4hxpdU4cRt+tn3Ksbb/UJYmXKWToLUjkWT4gnIYkfPMzCvfddt5YApovbD9g/y/fPpszcR2tDg/fpsdVordprq1xIEAcFcdQPRCkAcyEEMb9CGRC8frCcRpD48ig+JpNk8gBNjvtNjqkP5w7PpvJYCeHGnWKsuq8A2coO2r0+QESJ6P4RdW2IdlDOed3MknDmcChpzKOrm7jbsVIIK2HdP6wTq3za5fgpG8J480Aq9UtkhB7DJ3LEQE5XwiR3yHKfmcpEz7q8Ev4T/JJ+Ni3XlEXTem+zU3zTDxSWQLlzJeLsBlZtazep+1ZaqU7wjh/i6IQXzP87M83vvt5qa4pq9zqH5RJCOCiEo74QhkGUcIJIEAmihDjuLxvO5aC4TSD1/jq8WJ+luRNCSAiHhNC1QdjF2LjNoN+UH9XEYxHeyoXn17FAnHhHKuCHvUyxDx6cdcqTRHSARCdWeXKb9R4NTjkRGZy2q7SYP77pxtDNpw6ccilBHSLUl6byzstOL2gXCyGjhIUY/bE8ZcLQ10CtbcfunvutQsMPYQfO2ZRfeNwAzdBOa2+NRgpv5RLyNihvo768zVgoZ3Ugb+TtgLyNd+/tTRGhql+AitzVDVc9A2Q4b+jGUwt3+448pJE0PnqNro3GDl/BUsErZ8C9E0plnrEs5tvONEYkieSjJzmxSpDbrNSocI25EzMRQ+DP9TCahZlLTZ7ML2vJI3ncf4/PTVNOP3/e+RPGePUJjsLBs3fOpiIxsmj3vKL9btfrnu+924+ORAf/Aol8Q/Wz2Vhrcu+9J6cvn7z8UocOj4TIDY7cqC9yMvJjSG9wKASP4B0avHFf8JZwiHjqrla/JRyfxsU49XsmcZMTdsVnlSgnjsRxOBxdG44dVGe+Lz3Jgql0mL9IJQN6Y21GuqzWZuI0WtdNTsnpYJxOrNJmFy9TqajF+mnj0rXEZjEvNa9eAM85ZVSSOjypL04MxZwXJ10OI/CeYWE1cX4MUqA5T50p89giqOfQ11KhB185TqJUzT2YbXgks6va2urVxq03WHnfEVcRsGzpBY5x8njjAbQmKhvS+SwWENBBAR0RUAJKQB8v0HE/QPFid8ZCNvVkr+yNcr4jp+T00J26Nk47qB1hzIZBDLdnc/kmRcyuAhEdwZ1fxNULVnfaxbGWgfpMDsezTGMUyuWOceBluZz4Et9B851YpdltlpS+WQtSmXLx3SU17rkaaDmPaMn5jt5Bc70sbSYnz+R5wJ5HpsLTqOPxKzFS20jJNmk3XOXDG1kbUCtQEM3z0623SP7I32Z/ox794ePP1CeDZPCwDY77MYgXnDz7Mq1rRzLK2wwqSRAJ4mOH6NpA7PCtr8pNZjF/D0ZxDcv3aUS5kUgeCMmJVW7c8vg81UoPjqpcQSDHVda6zAs9Pg6bE2m3o6RJQgcm1DTP+YuuZ2K+CgQer8PEmdcYzk4tbSOtze7qpt7DVhe3Hzxx+9HMqr779oOayw55E0+YO+Zlh0DABgds1A8wFvJo9SspI2UHoWy8e2XvskiV15lXqXE6HjWSgFcIanOPR+JI3CMV59qI67DakodtWWvBaG7qK6bCUgpLqISQEO47wolV2tvqK1rqvaoi6qsU7+g5U1FYzm9eXKaSRtK41xpNgxy/6HIQyK+97GXm6qNAOJseesNflvmrX5l+1DkPoinN1vbxstTurcgumiyhfokdC5wL5jtT7/bDIjVZtT6K1oSn6v3JySI1G86PjRAPDvGoH8QZiTJwMbDhrC0DfDZAfIkv8bXhO34kORgnb+U/hZHAp4mwQO/qK18+BSxpVTYj02SaTKNp18Z0NxNxVdIx1pFYzkXewIpK3rbM1hBxHtwMZ9DgBMql8vE/QSfoBw59YpW8uxhxqJqhS9yGIL+rzlX8lZiDLfj3LMVX/1wQdsJ+yNhfmkagftnlqJzfspBlU9eBPb9ZPi5e8Q5xTRuk99lliXDKpgwVXYS3H3w4V3qCjTZaUwuzujS2oWMWXqj2idngmI16YlbEPWkjbQejbdyTNjkHbRSEkVjdJHJQknwqWp2T+kzuoQp48kf+9t6fa+Ovg1LRuzxaHRbCmUwE3uStfpEMmn2SgumP8I2aa5JBMrj/BidWObCLKo4MeDntnoGHvlOS4likwsIPaSSNe61xfGLolAQLOrwi9RhExerTFVyFLjE4sGLoPYUjmvJ1HWp5ZQMbcw/efwVg4AkIFWd2+wEIaPVtaK21w5limGnUSSwPhBwOyuFo9w7flJG7PlqIjGQ5mB0mQBnMx2STbB6mzfHubb6uyJM6MX5XN/LRWsVEc8yAQFTJxBy+dBpFosUIskSUiO4dUdeGaJeD+1S9XmpoaN779GQvvoa3I7JKVodsdWKVTrc66k8ZuHBSYxzDx3xt2ywCZQ9CGuFPQkno8ISOTUWhcac3o9mIWiJJIVBm2bAJ3E/XM6collj5u/d+S3dvbz8m0e3HWOC4WUJfg93QTGt4S9XfLYnUP2Ohs7fMD4r4DY7f6JHwY3DjeAU3nSSQBB6WwPHuBeZ3kt56HF9mMDZqzBYSSAI5SJCuDcgO6ztGlTo3snwT8igOsMdPsRbhJJyDxDmxypZbLejgWEFZPlSjLwe+eutIzbQlBWYVnmqY31HouaRsSmCHDtY1TAgPCzq8vL2AX9KvFCVxUrkUzthZsJwGuvypFv+p3Mx++oOttlUyVbudC+fH1D/DifySyDDW+uYDaD/qumpPO+56uYiMDsroaPdGFYANfYHyOGYpuSW35FbndtyXWxW22AtcBe3cACdbUW1ELInlQbB0bVh23EFoPX3KXgUVJs0+Qsl1KENebUdwCe7BwZ1Y5dMt9xZSYXq5lixlaUh36asd5Kfcx8braoJMkIcPeWIqOk1OOp7pSHb4g5/x1XXjTWv4kVpOcWS/s/rY8h5EU8rkZsah5av7bi1KNaADVS4hT4PyNOrHE6a4wF8EUUKmyNTATI17MHWthudRoZq/4qWZKazEEMI5ldsRMSK2b8RcG2IdFGLy2Yiqt2BZ7EY6GHgbBecEDkFNmDmbwU49NT8f9694FBNBIriXBCdWWa6LEXhQTDG5Vxa22krKHK4yIxynblaZwU86oeRH8vZU3shUAxl1eH35NxY4S/y/2XDj3JkyOKa0ZuE/V/9Xrmj/wP5B+y614dJF6iyX3BOOnxqGuTI21r7/Gzak7fOWLyB2g2I36ocdX05ZdJmSPJJ3qPLGPclbf9aGi6NNVl7jIjl8DmkkjQPV6Npo7LITTC1g5bPxIpprVlY/O1M4dUiXTJLJIZucWGXIbRZjymBd/ZIxS3U4mr1agoyLIb0SS2I5DJYTU6Vm0vHTdYbDN2G8wsmDEFmG4gw4xc0JOvIV2zxpv//O65XRJHD8ecTNZVFTQ63ZYTM6dfnnhG5Q6EaEjtARut2iG+8e3et6iKoJRPGQktVNbOj/Up2UKnsemEc8USSKg6Do2lDs9CWlust1aeZBjCsr2ZdoyCSZfPQmJ1bpceuD20BYJHKSRtXhE3tbx2m0kYlh4OJsH/WajWqC21ZtSCpJfeRSn5tKNs87ftaIM6/BGfQTOAt+M0NerX5LuMyRuHI+3Vu75xgPbaJeN8XB6XBfvheAVt+52FQ+rTfdfma5rCHtrHKVZURxUBRH/VCsTCpOIkkkiSxFjncv8l0ZpmuTGUMYbzLTWJlYEsthsnRtWHZQ8cmf4sPCMBKrm4ThyxZ4W2gGhLd2csOZnECnxEQ6SecgdU6skmYXL0phzOYzlevJaMacqZq8rM4tTimUkA4U6QtT2edFh1e237KwUaZM4ziF+DGXQ0PcyEbhA/deAvySTVW59WzKpqnhNcZNrbWvuKqWtEXXchEBHBTAUU8AjU80SCSJPGiR492LfF2L01qPHq669KCHDX16MjBt+vQQS2K5TyxdG5bdd+/Jja6b0/TuieA8evW17J+TEE/iuU88J1ZZs6OePlnkN3r66MAYZygPN14Wk1kyOyyzppnKJy87fmNEjYSNA22ns4gbZtho85ZIux3WO9TNBWx14W0Ypmpt761pyd3rYBULiNWgWI36YWUYIp+ckbNhOhvv3tmbLDolNgzY1Y39G1cqwAkbYdtDbK4Nti7nEM/l1Tltfqmq5RSKxI7YPTJ2E6sct82ySjEyL3ZT0zpo86pUq5mayB/5e0z+XprmXXrZ7YSki4UoIjV2piKepSKJjVOR4uo2vO6/45KZ+hCgwcbiwhN6asaWWoPLmtGRqywidINCN+oLnbGEQgpJ4aEpHO9e4ZtKlKpeMKrGovrAyHje2AmmGvFkkSwOxqJrY7HT6ksF5po0cwmmspJ9EYZMksm9MDmxyo/bLM2ogK0UZww6jF1cNl7lEk7COQicrmsYgRgW7OoWchZ4q09r9ctTVAtpMebwnyI92rO7367X4SXC8a+Z719Hwl+ktx838VtrsX2vsrwhbb+y6kLiNyh+o774QbRIAJDWPEYKSeEhKxz3pRAAJLAJxItBSjZhFNwr4ugZiQxxgkgQBwrRtYHYQfnmTZECZex6//r5n6ubZSDn5cb7wTxb6iqrcg7vYu1adm1XyiGn5HRPnE6sEub2SzpZwMIaQs4rZbh01c/vvcaYZJLMocl8aarndNnXG5KcnOEeX6ubw7n5EU4YvhF4njZTpZPwZRjL9TE51la2ArndxkqiYIL5TuB4tx8dNjfh3NT6fYwyk09GNodnc7R7m6+qIfq7uh7CSTgJZ45zvHuceNd5zjyeRtkoqasbvIbkG+Tkl5vyLpVJQ6ubiCVpxMkreT0gr66N125KRRio/upGjWRch4AuZTzr7xKLG1I4EcpTcxcEl+AOGe7EKtFus3b0tedcOyx/1JllXDMa7ZjHCpF8eSsPf2JLbA+E7XPXUFh67nZ4ffwNd0IerX4NROwsAzhhzJMTCCRBGsu+bktYUoOz+lk+LJU/++oG/tWfYdVX7gJJJ5W92IDdwRFU3sz0z7zAuXgPS5njp77jy3qxs4STyS7gGp89NQxrbneYrY2rRnXIyyWkfFDKR70ohziFQ6iFLgQKPrEh4SSchG9T+Hj3wt9WAnaZvShzxaN4dbOB2XepPOdZ0BNtok2076Dt2tDuoCT2TYHEWTIH5CxTX1alsUPUEiJMmsICWPMuNxuyKLpOzpc4Oqv8w6Cm/plzIk/kifwm8hOrbL7V0aTXiFbjN+ZFhUl78d6srK1+0fCT7PCOIHaSxt8T+jNAfwboz0Dtz4DpLb7nXb7A8JbBuYMzLMPjaZbGF0GQTJnn6bpI4ypJsZGV4620UUp9K0V6gSduPzrXT28/wBZT5t9+MPjc3Pw9XGLTT05fPnn5pV6nOjTyOTifo937fFOGajYIbx7MhJNwEs4S53j3OL/BgTVFnEcwXN3ipbQBT3XoJuHImVDUVpycktPDceraOO2gvFVGsHOXHmUHVoTAS7Kp7oWfwM3r6rekcMssZ7MnuSR3EHInVhm2ixnuKxErs+eGK2HDYFBqHKlEJCmnK2Tye3h+vzD1/fqiy2fGf2OBwxxoUf6MzJ/F6x2qp4JnMw/CN4GMKrewHz7x4Q2UMnEJ7OYiSIKL1Hfgv3qRmjZbM8waeTI+gf/VOawcBVEcHMVRPxTDII5Xn/BxD4kkkSSyInLcV3JMfe6oAU43ucwuLdWKVcoEk2AOGqZrA7ObNwSVNjlZdp44MaDxHuwuq9gTAOODsichPQCkE6vs2UVpqBq2kqqM5rveBMxtVxw1mBNX4jpErqZp3txO57rJE2ck4DBMVVN0KeB0yy6CCzgL/kzI4RxbktxOQ3WaiYAAvGb+lKcbxn2rt96+Olu2sKFEWz0M0jk4naP+dOLNaMijWHZLIKpElahupDruh2pxnYpez+AacJ5pjeM7R6zB/ujXEIMElaAeDlTXBmpHVaRNKORIU5IQ3iPCDde5vC9tQstWxvtGuZDcktsDcDuxSrDbLCxh1M5FDClWCphyls6CNHQqKTbePEw5gMrXpCRLWAeJ9f8BZ4Vs8jrHAgA=', 'AUDIT-DONNEES.json': 'H4sIAAAAAAAC/11Sy27jMAy85ysEn5P6Hcfd66I/USwEWqJtoYpk6OG0KPrvS8ttFtkLBUrD4XCozwNj2QhXpRX67Jk1xXG7ETaasOWflFEeHChDaVmld7pZQSsJQVlzr9pw6EPKKf1KTIQB7meo2jM9ZLUYsRZVh/KCVVUKkM2l6Np27BFLKEZBhzh37aUe20vXNEJCU3V9XWHf9GUxZIl0ABs1ctCa4zuIwL2NTiBPMnkAN2FSH1zEVGDwxr+L/r3uo0IMM5qgBHcorJMouVSg7RSTISNov3OMDo2Y+arw5vnwwcF75QOY8NCILFmRqDYYygcCcsER+7cADsJZ7xMf94tWj4qX6BbrcfNMWBOc1RolezP2Zk47A3tJgtgCDpbZgUfm7BB9MOj9L2ZsYMpIXJCCCYws+ClEWl7cd5d6WS35TMHGcFdncEW3G5rGuAtzOOLWOLnzuq99DmHxz3kOQpMfRGWnjyfrpvx3eT6VZd3n2fE/pHtXa4LA4POqLtqngmKZEexP6nNFIto606cCMaM77YaTC5Oj70mL+nEA4nSlEdNExzT4DdU0h5x0qMAkVSmt94EPX4e/Pt6qWvQCAAA=', 'CREDITS.txt': 'H4sIAAAAAAAC/zWPy07DUAxE93zFLEFKAgvYwKotQqp4LFoQYmkSN7FyH8G+Sdu/5zZVlx7PHI+/pU8SA+nx9pWk7wUWR60ZfCA/OLYCq1W5/Cm3i/K+uitgzBhYS437i/Vr8wYKzWXsxFLUY4UlxdEx6jgIN5hYfymJx06jhyVq+QGfm8X64+mSXD/bDHLsPUE5kQRuKrxnUQhkltEU0iNelEPdZXRIfEgjOfyNbKdPzoSBlIZOydhwXXcUWrabAiEmhFxiYkzkpKFT4iwr11Gb3DMzc1WbV1ZhS57Rh7gPSKQtJ+zIi5MMlgDO1iPmSxJa2OAkVVf//DIej1UBAAA='}
HASHES7={'mandia_stage7.py': '6044fb5633f0d0438000f005849f78db20c3b6c5edc2b693530818b53322ea41', 'teacher-grounded-qa.jsonl': '3cfe3c27ed8e221cad480755f9ee1a0fcee1c67583f58744cda427932e94910b', 'AUDIT-DONNEES.json': 'e20d55fd41faf3bcdfcf717de2b08813cce182b709623f02c50ed8ad2e4fe99f', 'CREDITS.txt': '270242f29e6f595a52dac0c7bbab715c8e7be232a9cf04ba93aa49a8f036c4f5'}
for name,encoded in ASSETS7.items():
 data=gzip.decompress(base64.b64decode(encoded));assert hashlib.sha256(data).hexdigest()==HASHES7[name]
 dest=ROOT7/name
 if dest.exists():assert hashlib.sha256(dest.read_bytes()).hexdigest()==HASHES7[name], 'Frozen source differs: use a new run directory'
 else:dest.write_bytes(data)
FROZEN_REPLAY={
 'mandia-compact-s2s-etape4/parallel-prepare.jsonl':'dc2bfd9ba783c5edec159ec35bfbb66206598520bef31f960da3ad92cc240cbd',
 'mandia-compact-s2s-etape4/replies-prepare.jsonl':'29334943cd507b8ffa9385b530b9625aec82c0741aeade3575a30c7df7e2caf1',
 'mandia-compact-s2s-etape4/data/waxal-baoule-text.jsonl':'4f7730d32d3324d2cc819f33d7633b2d59e01f81c8e2f38651d0377050ec69d4',
 'mandia-compact-s2s-etape5/quotidien-prepare.jsonl':'370d3bb631a4ad4765fbe3b15e0f6019c0470ce03c687f87ac71f6fda6b3048c',
 'mandia-compact-s2s-etape6/dialogues-prepare.jsonl':'9375c6a0918a5bf8056cecd42df0c8aff90c3439220bc45bfe10e8cc700a2184'}
for rel,expected in FROZEN_REPLAY.items():assert hashlib.sha256((ROOT7.parent/rel).read_bytes()).hexdigest()==expected, 'Changed parent data: '+rel
sys.path.insert(0,str(ROOT7.parent/'mandia-compact-s2s-etape6/code'))
sys.path.insert(0,str(ROOT7))
import mandia_stage7
report7=mandia_stage7.main()


MandiaCompact — étape 7 terminée le 14 octobre 2026

Deux continuations de 400 mises à jour sur L4, architecture propre inchangée (13 782 528 paramètres). Parent : CE étape 6. Contrôle : question/contexte canonique ; essai : reformulations françaises rédigées par assistant, cibles baoulé recopiées exactement de 40 exemples du TRAIN Wiktionary/Kaikki étape 5.

Les références sont connues dans tous les splits. Ce test réserve seulement des formulations françaises de questions avec contexte : aucune preuve de généralisation à de nouvelles réponses ou de conversation libre. Aucun nouveau dialogue naturel enregistré ajouté ; aucune validation native.

Sélection gelée avant test : teacher_french, pas 50 ; score validation 2,67705 contre 2,73884 pour le témoin, également pas 50. Les pas 100–400 ont été écartés par la garde sur les anciens dialogues. Les derniers checkpoints de reprise à 400 et les meilleurs poids à 50 sont conservés séparément.

| Test de reformulation, 40 vues / mêmes 40 familles connues | Parent | Canonique | Français enrichi |
|---|---:|---:|---:|
| chrF++ | 14,114 | 29,347 | 32,592 |
| Références exactement retrouvées | 0/40 | 4/40 | 6/40 |
| CE ↓ | 3,296 | 1,460 | 1,342 |

Les anciennes pertes test se dégradent légèrement : étape4 3,66286 → 3,68805, étape5 4,19511 → 4,24213, étape6 4,98540 → 5,05705. Les 14 anciennes réponses simples apprises restent exactes. Il s'agit d'un compromis, pas d'une amélioration générale établie.

Deux diagnostics avec question identique et contexte différent : le bras enrichi donne la même réponse pour « pain » et « sucre », et ne retrouve pas correctement les deux références pour « singe » et « aigle ». 0/2 paires entièrement exactes. Un changement de sortie seul ne constitue pas une compréhension.

Relecture technique des 40 sorties : noms de personnes et objets encore confondus, réponses génériques, faits omis. Exemple : contexte Mariame possède un mouton → « Konan le bua » ; contexte Aka mange du pain → « Aka di sukla. », identique à la sortie du contexte sucre. Ces erreurs sont établies par confrontation aux références sources ; la relecture n'est pas une validation native de grammaire ou de naturel.

Budget : mêmes identifiants tirés et 231 414 tokens supervisés par bras ; 732 256 tokens réels pour le contrôle, 747 192 pour le français enrichi (+2,04 %). Temps incluant validations et sauvegardes : 111,19 s / 110,17 s. Une seule graine ; pas de revendication de gain de vitesse ou de significativité.

Méthode : augmentation française assistée par grand modèle, pas distillation de poids/logits. L'assistant n'a pas produit de nouvelles cibles baoulé ni de raisonnement privé. Les articles de distillation motivent l'essai ; leurs gains ne sont pas ceux de Mandia.

Les 6 fichiers lourds totalisent 551 778 812 octets, tous sauvegardés et hachés sur Drive. Aucun fichier de poids enregistré sur la machine locale. Le code, les données préparées, audits et résultats rejoignent le dépôt GitHub public ; les poids d'inférence de cette étape portent une étiquette de recherche QA. Aucun Space créé et aucun backend public d'inférence connecté.


In [ ]:
# Verify the exact public inference sources against both real Drive exports.
import pathlib,json,base64,gzip,hashlib,subprocess,sys,os
ROOT7=pathlib.Path('/content/drive/MyDrive/Mandia/Sauvegarde-langues-MandiaCompact-2026-10-13/colab-runtime/mandia-compact-s2s-etape7')
PACK7=ROOT7/'public-inference-check'
ASSETS_CHECK={'inference.py': 'H4sIAAAAAAAC/41UzW7UMBC+5ymsvdihwW0XqERXRkJVbyCQWriUyjLJZGNI7NR22l3eiOfgxRjHye4irYBDlGTmm/9vZrFYvLOqIqEB8ghO1xoqYp8Mea9MpRXRpgYHpgRSNlB+7602YUWMJQ46G1BqKyCwgXII2hq+WCwy3fXWBaLculfOQ9Gr0LT6a+G3vmiUH7+/eWsyFPCo5Np4cIGdFT44NsH5R3wzKWvdgpQ5AjGNcEq9K2mez0GCdWWT1c52+PkdjP4BzpNJeTtLEqAbK5IeHpb4zKBU583ypsDnyppar4s1GHAqQJZVUKOdNiy/zEgv5pr4W7ceOsznY/xzLF/1XFWVVJOY0efPn0Cvm+BpEbY9iMOqCgcPg3ZQiVs3wDFbTFXX4MMR42Pw0poAG0Rjumpog6D0GCxh/h0cR0qLsrG6BC/uqIO+3dKCBqeMb7Et9H4fKCmPuangER0c5FT2AwKV6PnYwwj2LM+IszaIv059NfdDRN7wFgnrGVN8FhPrRi+ndGr66azh0YDm3IGqZKyfIXeI8pFvZCIj941avrpAf5N1Qn/dBsD8ct7AptJrdMZyIWbHd3Moen9HkwNsC/1kYNNDGXCJ9vtCs0ROkXLcETUltzqezYj6v0x2DuWcSHYQXYw7MjZtX2LRqV62tlRxa9NoikklrWm3Ezmm1PbOjgX7VyooE7tV5HEVx/GyuOwjPs9XcTSC0atEZCCXhJ4oPvH6hH4x78F7tZ4VUZoTXZMdhkDrgSTNSlfI25dJH9ksZp4m1Iv7E4zL8aqhjo2+OJqc3C0xWzRqwTD8z9+cX5xdOqXR5LNqB7h2zjpGr01wv34CCc72pLVmPcS0OrXR3dARtEnT9kQ9Qkl6xNZ6g0cykOsPNxTplzZjmkv6wcmkj3wVE27F7iyx3Vliz54dDqIchfQeeREs+8N6HLb0AXdVVroM7NBuL0bbCQ+Pqo2r+KRDk24q3x1+GSF4/xx4XONCivk6stG2wE4V58USGbWRqXDx+gJ99Q7DxQFjZWOjk4N4u2sipVEdbjhORsp4YKWkl+nQZr8BzthISJMGAAA=', 'tokenizer.json': '...', 'src/mandia_model.py': 'H4sIAAAAAAAC/61Z3XLrthG+11Ng2MkMacO05KSnqRplkp4mueg5mTRpc6PRcCASklGRIE2AspxM3id+Dr9Yd/FHUj+nJ8nxhUUCu4vF7rc/AKMoestkIdjrumpYrsl+SuakaWtd66eGk5x1ipXk5TlvhSZN3bXE0lN4EYUiQgotWCnUy7MirHx5ZroWLa+41Onk76146LgieS0l/s7J929/+LZuK0q+r7/7ipIfHsU3b/5DyTf/+pJwjXT7uuy0qKVbGTSANTVPJ28YPFZrIZlQMA0qNjUsyglXmnSSk/unptb3L78qGDo0L8+tQCVABKjKlCGRdbfnrNMvz6R4ea5qqVuQkE6iKJps2roiBdMsL5lSoKyomrrV/RAlTBUi1xM3UTF975913eb3VoR59MxSDgZTKf34ppM57hJMC6p9PZl8EZaZmP/kbV3w8nUtN2I7nxD429c5W2dK/MTnYHRNFuST6V9fmalCVH7s408/MUMle+Kt8qOfmrF7zoowZDl3+2w0emdGNxuZDWT+5ZXlB+9oftB++M+zOz+8z3a8lbwMU2aCaQ0OgF1mfM/bp6C23Q5rBZMgTOkWBqP7p3UrisjMacEzXq15UQi5Bd3WdV0Czb/bjtv98g3JMnC/zhB+WRYrXm4SayezMjgQrIyjKWyEfGQfzVbJYkGmZynttKP1ljlLHgfJt7cD3gSY7y7Ld1sGM5Cf/X4piYKZ8MUYs5blU/TLWRkDY5PPF2RGIBjtzJG1zezEgclFXSxlCrjqSu5sZQ05sCFFLA0MqbqGt3GSBqKkn8JFH7nY3qNTQfJ3rGUV10Bv0V5LrmIUl4S1NnX7yNrCLXUYLpRjrC9coLTqodXxId2UNdOwfFM/xndJWnEm45sZJTvOG5C8QEQk5JrM+M2rXrOW666VpOcnV1Z+kuoaRgvMbGaw38JwJlit1qx9+vBGa/lWKLBTtu42G6CMBPh00/IHAMAsnZJbEs+m8EeuroizJZhWbnk8NUtRArZwW7s1BoYMB9GOQqVefM1KxS/anJJ6s1FcD/SFQBIIHBXs75azlJ7j+pBi9olv7mC9gu9FzhdgMvMAA2i6hWU3un18128bpBnvhpWWc0q+BYCsvBe8CZY4Ssl8FXjzGjKvgphZWDEQAioeeZK6CSAaT/TrU7IG/sMyTVNKpvP53Yr6txm+HWPH7kNBSt7FMbuCJW/WVyAflsKf6zUOJcYbi5sZesOEH9rGo+ebtgYcFF/6sHwfHOWb7W/Akck5dJyuaD+FKRw2DTI9JT72hPgGJLe3gWIs/sGG9RshOWtjRx3YwKKCqSHYAuPuMqNf/Wqk5Ttk7T+grLrTv3dL0Gwgfl1GGC34jkgD/Gi+sIi2MbSACM5Zfs/dKr2z10zn95SUXG41/GYI11Tds4YHEnSI9Ux8SNK94I+x5XJMA1CMNUw1xLOC4OPxjA7CcucF7t4hMADm/WTuvcz9h5P5QHtV0RPxQ8hidDC6C6N9amMKfd7XY7GxXiFCQTeoTRLqnTBgMVTL6SrkvBERLLUP2TKHRBx7erpLqMkKmCVP5mcrug/zQaLkj5nVakFwF/sE9TQ4IRxgYrQM1H8ib7ibbKH3BRIZ+jjyN5JzjdurGujFTQNsumnsOtkakjA0vM3Lr4ZLdQJ+/3n7o+3rf/jHd1+mPTawK9tBrWo4wzYL6lXJ2Z734Fe3t2Nn4rZmAySghP0fkVAxtaP+CLIgNpCsf458agaxFbIoI59DWh857EEcFzeHx6NClkBlsoXp2ggde/1EyK4viSeCjLqDQuZ3hHYV5LMF6BTmnmDw69Q0KRAKtc7geFN0uc5CVweY37V031IYkRnKWRjzFHgO6oBhMU2nVKjMmmthf5LRCk9HAYbdpBbbru6wop7G6s3spKnyiTR+AngH3IaaB8/Fa7MyHF32H77kuc7XVrVBLzymAuP992ym978X8zyKtJyo/6w44Txal26xyqvF/xW8Bctk7+qWf+It9DVOTnKOG2X/Dt5LVe+Syn+4knU+VaMbsAhcyOrDVHyahn0+7FJEmd1hZ6NtmlD3NEvoABZDuCr+0HGZ8zNJmnYuA8/G0REAEHvmi6pvrW6uSxTbqhZFfLg69vSwF70eO3I4lRynAVF2EF9XSHy2Rnj9IFfN6U08NsF8ZSw7iBZIhraMjPjm09X7BXeal+Cd+GxNcnFvr3E+fLg7O58Hr7uluBxzXfP7eYv6UZ5we67fEDqDzQ6NjOJj5+mwU4iV5Mppjs/eun8vaziGvJ9xKRGy4If3s3EoLi6h+gsKqK6DWwlStyQ+nnbXF6bmxmbJ61nyEVId30NgqT7O4sCAdz7bBxYhqno9DLLMJUg0ZpF1W81o/3yH3bi71fCJj56MjGVU4gBcJ+cyhOQZLY5rGZKN5QEcQJ7Dfpj+w/mz4KVmlPTBblWPezsAOKhNZe6EbgT12uE2D9dGztGY17sXdmeQdoRRr7FHoL3yffP2w0c4UFvwjYfD7Z+Nwa/8q8VhuAml5x29xoBxtdKq+0YoHS9tIGGUiAR9RATexdlODiXZS9NklZxi7wzcTg/jrkc1iRKIxteYFlXn09FgRxczEmua0h09jS37adyJhIYAe2PXGOC+rO7wXmRhHPq8cZkFVZEk5bJQj0Lfx3EEWMMi4C7G8FISAGMylh9KjoSY+iRT1MoYi5VZHJYEpBbYm97d4oV5am727q4G5rYb+QIBJ3Jgua+LHlsoM66GkHNaCyUksEA1c9M0DralQ8jQ0Mwdq32sspXjNhnUvhTVQjbQdgtz5YKxot4zvEF3d58WJPhmBiq1D4rUX7aPNG6ZAAz9yMqOf9W2dRtHry0Zx+8ZcF5RL89kDmQ5fjXJRUtKBqpXEL6c1B2+bV+e5ctza06FaTTIGYOKr3zaCeiNg66QZZerEfQEJSbgEHNcdhVvsZYNAvHI7Ic+t5n5+EB91LjVXUPomhYzthSrC8lu1BwpDBPAsu30xhkxxDJkvDBR1luhlWm7ShuXBzreusNDEroqE+ljFf3wULK/hbYr0F7F40bKEgSUhcDJwIdSH3/a8F1EV8VNitYuY5vLmhDzw2jv0VtwlbeiMSXvvMyfsfBuxDaa2w9dsQdjQqNeZjQfL+LUTOhJSois+6P5cm1LPqq5Dmra2RWN/CfEnwwoo3kEGbmoKwBkTRrQDVAveUGsH1T0y2Tyhe28hdzwFhvaDEIXOtQJ7nPLpUEgghVDGg5pKGUjAHe8VjBM4Rh+yNAhut5xqRavPqFE86pBvq7lGPbOOoY/5XtWxt6SeLjH0gJRZ8WP7e2O/xOPOqspdBYKwnVpFVnR4W05dNdbf3vg7g4mPTR9dgFRZjXAp80r5tuHIfQ7LnzyAdrlytwlxnZBS3dU8Soh47EhqN3PIP3cDGQM02dpL8msisspvZmt/BeJQGNkApEAdCB9ytotrAc2wgAY2BvvQqY2FpDWWqXqSi1kXQEwsFGuNxp5Uc7tgNUf5Ohs0MQEe/hkYDQZHTydbguHiHGCWrec7Ua15npGPiO/2V5jqRfcOcbH0gherY7wYLs9NcqAAwD4P+f9a/zyN4jrYI+JCRDDna2fQGDsMtJ8cpRatE8tVzrlpfmab9oTl2vcuV36zeCY+a4Zu5lRfbatqu4aqOHWzUsztLJAGF7FJpP/ATYgikqRIAAA', 'src/mandia_seq2seq.py': 'H4sIAAAAAAAC/7VZ25Lbthm+11MgO9MhaMPcley4ibbMOHVs98LOTLNtbzQaDkRCEkYkyAVIrZRO3yWX2efQi/XHiSdpa6dNd8a7JPDjP37/AfTV1dUnKjJO35ZFRdMa3c3u0BzxospPjwUTNa15KZClIYgLXnOac2WXKRDRuuSSRZN3Ii0z1ki04hkspJpAsByxGtE0Pf2q0OkXlFOkykamDGVUKJRu6X3D0CovU5Q1KDs9Wh7R5C/HleQZA1VeobQU+zJvNEM4QhtFc6bQczRDH/76PbpF8vS4Pj1KJlJN/1qvRpOPQJKCTaWiolYgVdSoalY5Pz0qOJOyumaaYMUF5bCLRMAUkFCFGsGQKJs9o019ekSVhOfTI5h4dXU1WcuyAOVrmuZUKRACripl3S0RRFXG03riNgpab/1zXcp0a1mYR39YiN5iJIRfXzfC+JHmwBS9t0SFiUVSgKdyT/jTp7sfS1kQ9FNZU3kk6AOtWfbW+OotuI+guwf+4ePfJ5M3raIT81sHHCjWfDOfIPjZlyldJYr/zAAF4LMYvbr59rXZynjh115+88osbRnNlF+0VLt9MlidmVVm0CGTnB6ZHO1l7NKe5bZei6Qn94+vv7FaUskhrnOkagnLwdbAJTB7Gi7JjkkAnz/2td+o2aFuhX/tzJJlVTawvM5Lqjei6cT55nsAidABwEJEn8qsyVk4dzqvUZLodEgSrFi+Jul64/b0j2oqJnEYtTRhtwXU0ZaYP7u9e8hiOB8Zx2lOkfeieQEHXF+3+0NG9zGo9pELRiV2tP4MWXGq4vc0V2wkfXfhkNfnmdXn6cP7/+UwOPq/URhCxGKLbWxFhG0YduyobAgOvQCsiCBJfIjUllasXZWsbqRAdVPlDBu84UMY7Tl7wPrAKCZhVEsoU1BDGJ6SWYjWpUTmFEAIWUV2lnYfdgoB1QOVmdOJ2IIX/1gKRvZQOzP7qKC0utVyvVasjm9IStMtsw4gttA5b3zOrvvYguHcmqdsaY9CxQPR7Stf+wrNFVTBGmkNO/Emw8k+NtobYvvQ0SIGCjuYmcgYbp08vX2BX0t/CG/vyS5uw47vnX9C0q3t/NqA0SV1hqJagw3d4mZp/bh4MVveajVsAU5pjbGnILuQvJiF5GxruiR7s9VKKKjamYj2vWlCPtDIkJnlxZwYAJhf82V7rFEscfG3fxDUfKv4MFSjXfMgvpsOjTbyrPoUMLAB90W6wGNtVsb2HNB5iOxDuHC6oD/FaHDGOtz5/bk4P2ht6Yw4xu8jlUKvzpKsrBNoo1mT1gltayqEOZKsYrSGIlkzmTO6ZzarttfXbS5OQ7L/QjrgLRJjrv5FuPJe7BwajmuBL0v4OMqQSPcLvmnKRuFeVr2YhiHBgMB9aCMAKWsxr80Pfeuw45D8M4w2u9+he9waNQX0+Gnsmr2vnr292dnesIxq/8RdU9NC7WFos7GdEHqLujHqYv2DbZCWp31+stgZWPcsOUKxakXjzghIc1cM7YnbQ3x43krFx7Mw9Xe9yh2/GfALW9//wH6r7wkXGTt8QQR2QBgHesQITPTBJW4WiWM/hZgsNAz/8PKreGbBEWzuaTCMhnEE6WzoHl+exZF8dmHIu+CHeDQDmsiaGtka4i0xGg5xMWSXylKp/y9ybJ+woPhcc+wHCk4bamJ0tAdxW1XDQVPo9Se9NGh6Y6fMj6TlHXufjhDcCXfV0eg46nVfwqfd7HOL+0xj89tPBX+TTU/Mk7lz7DulC+Qwb/yA0g9A3A9Dx6OvytPiR7l8nq0vbbaOM5zgS8Fss9reQeG68ruVU6DSY/cQ6qxYsSzjYqMB/M6/GAh3dyNyOevcLSduFfzIVY0Xg15gslAPkokeIm1v1dyGN6RwOWLtLklj1oNSZwqZZc6HzIdXrDPmIFvHhXhJ+iX+jRWHVlXuhnPj5m5b6yNowUillbIogNcsqWC0KBh0dGiw8/Ekp0lAsUw98HqLcQC1I3pgfLOtAxJk5YPwb2E4B6dokQZcNE9wBcjN4uhmdq1v35G6lzWePbvg5ucXnOOQ+Uajj6eg37bMOphpObjoqQuqcsUFUIuU4YLg9npD+gDSL7oKT7ML+hbOFq92VyStuhbULkErmsU3PQVswtrNr2LY7aumh05DEFFxxDC7RDTPwduScqiC/6B5w95JWUoc3Nmpf88zFgX95B5mxdk8r6OrP+Ds2uA6F88PsVnHfiq4OHk56LXzQGu5jYq1HO59GyiE5y1CfVGPAD+4sdUyckP/dPmdLwOR/zZw7pe3dod1X6f0E0zb6vT4Hx1lZYW3+sq7WA78xUnrMSaagkkwBffTfJQO0B1bXw6cYBqa73DqQotTC74cNietj05WSC2sOvXzcsNruHxHub2a90uBb3WdcQ6vZyHFlg3RjX40Hdudy93f2eQ81tlesKKUR9eSetjyILztI8kBxmGlfzTsgUqlkldmhtGHerIcq3/q7r/mm2BuP+NhD5GQBF29CuaqKXAV6eBBOpmgdtWtX9fgmK8v0PeD+WJlpgybNu0JR7Ikgf/C+rP5whrMAyjiWVkQyGRUgY4AUMEyZL7+Bf+aTN7YSxoXa2a+gJrvgjicaGs3TFhoGWrv5VWpCIN/BT0kdbljQsXfvnaOMIQRA69Bj4SLXSwA+vb40Cp365ucV0AnJYf2FM7daGF1hOFRQUYt7OLS3yDhjw3QINpOk2G4ezl/a1QfcV6AbcsB41u40T1A23QJqL1us6Hrj50femhwOHYCrTIeYJp2gC9H5vNsOJztdS2J4eLqcmNxA1fIZWQ+NsJMAmgFBcCjTlOfmuZYOPqQAIxiCN18JRndtXsXPWGol+dOdjC3wpziE4MW85ysjrBgv3B4f5zVl7ljcuO3HQZV4sT74z49zatPlmfuleVM/weDmafwQDmdW31tDjbBDiZdLOvJvwGa92HWuRgAAA==', 'weights/manifest.json': 'H4sIAAAAAAAC/6WSz27cIBDG7/sUyOdsjIn54722Uk895QGsMQw2WmwsYHeVRnn3YjtVlVulHhk+fvN9M7yfCKkg6sll1PkWsbqQ6icsxsG3MK+gM3llr2R6G6Iz1dOmXiHCjBljKtrmRSrGmdpvUoZxA4jjhL4g0fQQ542qsddhyTH46ut9yrhuKEr3+gPdOOUN/l6OpWCdx6W03CDzbu2sD2/nvaM4a3xe804t8uEt4/aa80Y2vG0/y2kCxsXGAAYWecOUEQNwq6hEajRTA1hrrKBCUjUY7IRoaCs69SJNZxqlEZQeGJg/jW7Rb7gp5zVd6np0eboNz8Vavbs8z8Ggr786rmNJDQlTbcJj8QFM/ZkhYsJtEed7U/9DyqNbb9yIKW82jnyX/w5X8B/7HnK44uJ+Yez/jk4PqmWd0VYyYYBSbrpOtQIoZ7xjWspGc9ZQaMXQddgOlNG2FUraTjKpDRybtzfv+7BmN+94PaG+rsEt+9KrBHc0xC3kEeLVLSP5EcLokXyP7o5PZAmZlC+YSbAkTy4VpcWIi0YCKeExoGqBXNT9HbwzUD5ZAVvwCU8fp99h0opy8wIAAA==', 'weights/stage7-qa-manifest.json': 'H4sIAAAAAAAC/31Sy5KcMAy8z1e4OO8sxoNh2JxSyTWH1H4AJWx5cA3YxDbzyNb+ewRkM7WX3Kx2S+qW9LZjLIOgeptQpTlg9sKyH+C0hW9+nEAl9ipeWX/vgtXZ08KeIMCICUMkbnGoj0KK4/oTE5yWAvUW4UAlUbcQxqVqQlA9htYEdKrPPnNiwolIkm+w8tOqBG8TBjuiSzAw5V3CW5rp+fPrEys5Ozt/dSxBOGFiBkY7WIxfmPMUBcQl40I6IVnvtoZXtKc+LcrfKCTA2AEd+Vm6javvvdqM71c79f4XPE9pzSZ6d0+4ZEtZ1IUsy79w7EHIalVcGZRdXfC6OZZCi6osdQeiUsooxQ9KNkYbzaHBUulO8gYM6MKYqjh0x6PEj0ZzGJZyfUpTfMnzk0393D2TtHxVuR+9xiH/rDgPNE6IGHNNcxk86Pyfh33AiMui95ci/59Rav++jir5Mzr7mzb2cKc6MtVoZWpRaeBc6oZsVsClkI1QdV0oKQoOZdU15LDjgpdldaxNU4taadiWYOZhaP2U7LiWp6tQ58lbt+4l60CdUbN5YtaxKdgLJGRXH87Wndh3ircZZY7WesH2AoPVRNGUa2CIuInHmFq8kTuCqwcU/HVpUvIHtJ1P+3E+7RTwYv0ch3sbER2xU5hx9777AzEKauksAwAA'}
HASHES_CHECK={'inference.py': '51fbb50d6f85b14afda3fe75788bcaf812cd80ffa0fbea2f180a956a6328010f', 'tokenizer.json': 'cb8429dcf726da005d99846a052592c771c5210a46b99e4b02044687f9727cda', 'src/mandia_model.py': '629583e3a5a5e1c3ea418f30b3d2fab7d9e05e07dba5adc4e7563436ff957d96', 'src/mandia_seq2seq.py': '6d1b84f0fba29e716741c11cff71bca8b622c9d850feb392f9d77c7b9ed2f1af', 'weights/manifest.json': '077d421521905257292de29e686eb24a97b2f2bd7a5d9a1c441a524153267e7f', 'weights/stage7-qa-manifest.json': '68b7acc392816314b639c4779266627b1557366c6de45148210b95ff4adcf7d8'}
for name,b in ASSETS_CHECK.items():
 data=gzip.decompress(base64.b64decode(b));assert hashlib.sha256(data).hexdigest()==HASHES_CHECK[name]
 p=PACK7/name;p.parent.mkdir(parents=True,exist_ok=True);p.write_bytes(data)
env=dict(os.environ,OMP_NUM_THREADS='2',MKL_NUM_THREADS='2')
checks=[]
for label,weights,manifest,text,context in [
 ('stage6',ROOT7.parent/'mandia-compact-s2s-etape6/ce_control/model.pt','weights/manifest.json','Bonjour !',''),
 ('stage7',ROOT7/'teacher_french/model.pt','weights/stage7-qa-manifest.json','Peux-tu préciser l’animal qu’Aka aperçoit ?','Ce que voit Aka est un aigle.')]:
 cmd=[sys.executable,str(PACK7/'inference.py'),'--weights',str(weights),'--manifest',str(PACK7/manifest),'--text',text,'--context',context]
 output=subprocess.check_output(cmd,env=env,text=True).strip();assert output
 checks.append({'variant':label,'output':output,'checkpoint_sha_verified_by_cli':True,'public_code_sha256':HASHES_CHECK['inference.py'],'device':'CPU FP32','scope':'artifact loading and generation smoke check; trained/source-known examples, not independent quality test'})
 assert label!='stage6' or output=='Anouho'
result={'public_inference_cli_success':True,'checks':checks,'all_weights_read_from_drive':True,'no_heavy_file_local':True}
(ROOT7/'VERIFICATION-INFERENCE-PUBLIQUE.json').write_text(json.dumps(result,ensure_ascii=False,indent=2))
print('PUBLIC_INFERENCE_VERIFIED',json.dumps(result,ensure_ascii=False),flush=True)
